<a href="https://colab.research.google.com/github/Tucannsam/Recommendation_System/blob/main/notebooks/02%20baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
import numpy as np

In [58]:
#PART A
import os, urllib.request, zipfile
import pandas as pd

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"])

print(ratings.head())
print("Number of ratings:", len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


In [59]:
# this downloads the MovieLens data and unzips it if I don't already have the folder
# then it puts the data into a table showing the user, movie, rating, and timestamp
# i print the first few rows and the total number of ratings to check that it loaded correctly

In [60]:
# PART B
def split_ratings(ratings, user_col="user", item_col="item",
                  test_frac=0.10, seed=0):
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=bool)

    user_left = ratings.groupby(user_col).size().to_dict()
    item_left = ratings.groupby(item_col).size().to_dict()

    users = ratings[user_col].to_numpy()
    items = ratings[item_col].to_numpy()

    for pos in rng.permutation(len(ratings)):
        u, i = users[pos], items[pos]

        if (rng.random() < test_frac
                and user_left[u] > 1
                and item_left[i] > 1):
            is_test[pos] = True
            user_left[u] -= 1
            item_left[i] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings, user_col="user", item_col="item")
print("train: ", len(train),"\ntest: ", len(test))
print("test fraction: ", round(len(test) / len(ratings), 3))

train:  90032 
test:  9968
test fraction:  0.1


In [61]:
# this splits the ratings into training and test sets, with about 10% used for testing
# it makes sure every user and movie still has at least one rating in the training set
# the seed gives me the same split each time I run the code, to make sure it is consistenet
# keeping the test ratings separate lets me see how well the models predict ratings they haven't seen

In [62]:
# PART C:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
  R = np.full((N_USERS, N_ITEMS), np.nan)
  R[df['user'].to_numpy() - 1,
    df['item'].to_numpy() - 1] = df['rating'].to_numpy()
  return R

R = to_matrix(train)
known = ~np.isnan(R)

print("Matrix shape: ", R.shape)
print("Known training ratings: ", known.sum())

Matrix shape:  (943, 1682)
Known training ratings:  90032


In [63]:
#^^This block of code creates a matrix that is 943 by 1682
# It then fills the matrix with the known ratings of each movie and
# turned the training data into the rating matrix.

In [64]:
truth = test['rating'].to_numpy()
u = test['user'''].to_numpy() - 1
i = test['item'].to_numpy() - 1

In [65]:
#^^This block of code gets the actual ratings from the test dataset.
# It subtracts 1 from both the user and the item to match the starting index with python

In [66]:
#PART D:
def rmse(pred, truth):
  return np.sqrt(np.mean((pred - truth)**2))

In [67]:
#^^This block of code goes through the hidden ratings and performs
# calculations to find that the predictions usually have a 1 star error
# from the actual ratings. This means that lower would be better.

In [68]:
# PART E:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0

item_mean[has_rating] = (
    np.nansum(R, axis=0)[has_rating] / n_item[has_rating]
)

print("Overall average:", round(mu,3))

Overall average: 3.53


In [69]:
scores = {}
scores['always 3'] = rmse(np.full(len(test), 3.0), truth)
scores['overall average'] = rmse(np.full(len(test), mu), truth)
scores['user average'] = rmse(user_mean[u], truth)
scores['movie average'] = rmse(item_mean[i], truth)

for name, score in scores.items():
  print(f"{name:16s} {score:.4f}")

always 3         1.2516
overall average  1.1349
user average     1.0426
movie average    1.0284


In [ ]:
# this code calculates the overall average rating,
# each user's average rating, as well as each movie's average rating
# using the training data. it ignores the missing ratings
# when calculating the averages

In [70]:
# PART F:
def bias_baseline(R):
  """Overall mean plus one user effect and one movie effect."""
  known = ~np.isnan(R)
  mu = np.nanmean(R)

  b_user = np.nanmean(R - mu, axis=1)
  residual = R - mu - b_user[:, None]

  n_item = known.sum(axis=0)
  b_item = np.zeros(R.shape[1])
  has_rating = n_item > 0
  b_item[has_rating] = (
      np.nansum(residual, axis=0)[has_rating] / n_item[has_rating]
  )
  return mu, b_user, b_item

In [71]:
mu, b_user, b_item = bias_baseline(R)
pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)

scores['user + movie effects'] = rmse(pred, truth)
print(pd.Series(scores, name='test RMSE').round(4).sort_values())

user + movie effects    0.9545
movie average           1.0284
user average            1.0426
overall average         1.1349
always 3                1.2516
Name: test RMSE, dtype: float64


In [72]:
# this code calculates rmse for four different prediction methods,
# comparing the predicted ratings with the actual test ratings
# to determine which method has the lowest error.